# **Limpieza y Preprocesamiento del Corpus**

- **TFM:** Evaluación experimental de Recursive Language Models para el análisis automatizado de literatura científica

- **Autor:** Juan Antonio Jiménez Cobo

---

## **Propósito**

Este notebook tiene el objetivo de limpiar y preprocesar los `N_PAPERS` archivos `.txt` con el texto extraído de los registros del corpus generados en `01_text_extraction.ipynb`. Se ejecuta un pipeline de preprocesamiento que crea un archivo `.json` para cada registro y lo almacena en `PROCESSED_DIR`. Se llevan a cabo los siguientes pasos:

1. Monta Google Drive, instala las dependencias desde el archivo `requirements.txt` y configura las rutas del repositorio desde el archivo `.env`
2. Define el tokenizador `cl100k_base` para estimar el número de tokens de cada registro
3. Carga los metadatos de los registros del corpus, así como el informe de extracción generado en `01_text_extraction.ipynb
4. Define los patrones para identificar las distintas secciones de cada registro a partir de expresiones regulares
5. Crea la clase `PreprocessingLog` que cuenta con un registro de las secciones eliminadas de cada registro durante el preprocesamiento junto con funciones axuliares empleadas en el pipeline principal
6. Define la función `preprocess_text` con todos los pasos del pipeline de ejeución, incluyendo:
   - Eliminar bloque de autores inicial (texto inicial previo al resumen e introducción con la información de los autores)
    - Eliminar disclaimers
    - Eliminar apéndices
    - Eliminar referencias bibliográficas
    - Eliminar agradecimientos y secciones administrativas
    - Eliminar marcas editoriales y cabeceras de revista
    - Normalizar saltos de línea
    - Limpieza final de espacios
7. Aplica el pipeline de preprocesamiento a cada registro y genera un documento `.json` por registro con el texto preprocesado y datos del registro (título, autores, año, DOI, número de tokens), almacenándolos en la ruta `PROCESSED_DIR`
8. Genera un informe de preprocesamiento en CSV con diferentes métricas para cada registro
9. Muestra estadísticas de comparación del corpus pre y post preprocesamiento
10. Permite inspeccionar el texto preprocesado de cada registro y compararlo con su correspondiente archivo `.txt` con el texto extraído inicial

---

## 0. Configuración inicial

### 0.1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### 0.2. Instalación de dependencias

In [ ]:
!pip install -r /content/drive/MyDrive/TFM/requirements.txt --quiet
print("✓ Dependencias instaladas")

### 0.3. Configuración de rutas

> **IMPORTANTE:** Las rutas se cargan desde el archivo `.env`, ubicado en  el directorio base en Google Drive. En caso de no tener `.env` configurado, se usarán los valores por defecto indicados en el código. Consulta el archivo `.env.example` en el repositorio para ver las variables disponibles

In [ ]:
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env en Google Drive
load_dotenv('/content/drive/MyDrive/proyect/.env')

# Carpeta con los .txt extraídos por el notebook 01
RAW_DIR = os.getenv('CORPUS_RAW_DIR', '/content/drive/MyDrive/proyect/corpus/raw')

# Ruta al CSV con los metadatos del corpus (exportado mediante Rayyan)
CSV_PATH = os.getenv('CORPUS_METADATA_PATH', '/content/drive/MyDrive/proyect/corpus/metadata/corpus_metadata.csv')

# Carpeta de salida para los archivos .json con el texto preprocesado
PROCESSED_DIR = os.getenv('CORPUS_PROCESSED_DIR', '/content/drive/MyDrive/proyect/corpus/processed')

# Carpeta de salida para el informe de extracción
REPORT_DIR = os.getenv('CORPUS_REPORTS_DIR', '/content/drive/MyDrive/proyect/corpus/reports')

# Ruta de salida para el informe de extracción
REPORT_PATH = os.path.join(REPORT_DIR, 'preprocessing_report.csv')

# Número total de artículos
N_PAPERS = 55

## 1. Imports y utilidades

Se configura el tokenizador `cl100k_base` de la biblioteca `tiktoken`, el mismo tokenizador que se emplea en GPT-4, para estimar la longitud en tokens de cada registro procesado.

In [ ]:
import os
import re
import json
import pandas as pd
import tiktoken
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

# Crear carpeta de salida si no existe
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)

# Tokenizador cl100k_base
enc = tiktoken.get_encoding("cl100k_base")

# Función para estimar la longitud de una variable de texto en tokens mediante cl100k_base
def count_tokens(text: str) -> int:
    return len(enc.encode(text))

print("✓ Imports completados")
print(f"✓ Carpeta de salida de los textos preprocesados: {PROCESSED_DIR}")
print(f"✓ Carpeta de salida del informe de preprocesamiento: {REPORT_PATH}")

## 2. Cargar metadatos e informe de extracción

In [ ]:
# Metadatos del corpus
df_meta = pd.read_csv(CSV_PATH)
df_meta["paper_id"] = [f"paper_{str(i+1).zfill(2)}" for i in range(len(df_meta))]

# Informe de extracción
df_extraction = pd.read_csv(EXTRACTION_REPORT_PATH)

print(f"✓ Metadatos cargados: {len(df_meta)} artículos")
print(f"✓ Informe de extracción cargado: {len(df_extraction)} filas")
df_meta.head(3)

## 3. Definición de patrones de detección

Se definen expresiones regulares para identificar los límites de cada sección a eliminar durante el preprocesamiento. Los patrones cubren las variantes tipográficas más habituales en documentos académicos, incluyendo mayúsculas, numeración y prefijos de apéndices entre otros. Cada patrón se aplica sobre líneas individuales del texto extraído para detectar el encabezado de la sección correspondiente.

Las secciones detectadas son:

- **Referencias bibliográficas:** `references`, `bibliography`
- **Apéndices:** `appendix`, `appendices`, `annex`
- **Agradecimientos y secciones administrativas:** `acknowledgments`, `funding`, `conflict of interest`
- **Bloque inicial de autores y afiliaciones:** contenido previo al resumen o la introducción con la información de los autores, sus afiliaciones y contacto
- **Marcas editoriales:** marcas de agua, DOI, marcas de copyright,...

In [ ]:
# Patrones principales de sección
PATTERNS = {
    # Referencias bibliográficas
    "references": re.compile(
        r"^\s*(\d+\.?\s+)?references\s*$",
        re.IGNORECASE
    ),
    "bibliography": re.compile(
        r"^\s*(\d+\.?\s+)?bibliography\s*$",
        re.IGNORECASE
    ),
    # Apéndices
    "appendix": re.compile(
        r"^\s*(appendix|appendices|annex)(\s+[A-Z0-9][\w\.]*)?\s*[:\.\-]?\s*$",
        re.IGNORECASE
    ),
    # Agradecimientos/Secciones administrativas
    "acknowledgments": re.compile(
        r"^\s*(\d+\.?\s+)?(acknowledgments?|acknowledgements?|funding|"
        r"conflict of interest|declaration of conflicting interests?|"
        r"author contributions?|data availability|ethics statement|"
        r"competing interests?|financial disclosure|"
        r"ethical approval|institutional review)\s*$",
        re.IGNORECASE
    ),
    # Disclaimers
    "disclaimer": re.compile(
        r"^\s*Disclaimer/Publisher",
        re.IGNORECASE
    ),
}

# Patrón para eliminar el bloque inicial hasta el cuerpo del paper (resumen o introducción)
INITIAL_PATTERN = re.compile(
    r"^\s*(abstract|introduction)\s*:?\s*$",
    re.IGNORECASE
)

# Patrones de marcas editoriales
EDITORIAL_PATTERNS = [
    re.compile(r"downloaded from", re.IGNORECASE),
    re.compile(r"©\s*\d{4}"),
    re.compile(r"all rights reserved", re.IGNORECASE),
    re.compile(r"doi:\s*10\.\d{4,}/\S+", re.IGNORECASE),
    re.compile(r"https?://doi\.org/\S+"),
    re.compile(r"published (by|in|online)", re.IGNORECASE),
    re.compile(r"preprint\.\s*arxiv", re.IGNORECASE),
    re.compile(r"^\s*\[CrossRef\]", re.IGNORECASE),
    re.compile(r"^\s*\[PubMed\]", re.IGNORECASE),
    re.compile(r"^ACM Reference Format", re.IGNORECASE),
]

# Patrón para detectar cabeceras/pies de página de revista
JOURNAL_HEADER_PATTERN = re.compile(
    r"^[\w\s]+\d{4},\s*\d+,\s*\d+\.?\s*(\d+\s*of\s*\d+)?\s*$"
)

print("✓ Patrones definidos")

## 4. Funciones de preprocesamiento

### 4.1. Clase de funciones auxiliares y registro del preprocesamiento

Se define la clase `PreprocessingLog` para registrar las operaciones realizadas durante el preprocesamiento de cada artículo. Almacena las secciones eliminadas, los avisos generados durante el proceso y el número de líneas editoriales eliminadas.

Se definen también las funciones auxiliares empleadas por el pipeline
principal:

- `remove_author_block()`: elimina el bloque inicial de autores y
  afiliaciones hasta la primera ocurrencia del resumen o introducción en las primeras 100 líneas de texto
- `remove_section_from()`: elimina el contenido desde el encabezado de
  una sección detectada hasta el final del texto
    - Para las referencias bibliográficas y apéndices, se requiere que la sección aparezca en el 40% final del documento. Esto se define así para evitar que se elimine contenido relevante al referenciar estas secciones en el cuerpo principal de cada registro.
    - Para el resto de secciones, tan sólo se requiere que la línea de texto anterior se encuentre vacía.
- `remove_editorial_marks()`: elimina líneas con marcas editoriales
  (marcas de agua, DOI, marcas de copyright) mediante los patrones definidos
  en la sección anterior
- `normalize_line_breaks()`: une líneas divididas artificialmente por
  el proceso de extracción del PDF

> **IMPORTANTE:** La función `remove_section_from()` restringe la detección de referencias y apéndices al 40% final del documento. Este umbral puede no funcionar correctamente en dos casos: (1) artículos con cuerpo breve donde las referencias ocupan más del 60% del texto, y (2) artículos con apéndices muy extensos que desplazan las referencias hacia el inicio del documento. Téngase en cuenta esta limitación al emplear el pipeline.

In [ ]:
@dataclass
class PreprocessingLog:
    """Registro de las secciones eliminadas y advertencias generadas durante
    el pipeline de preprocesamiento."""
    removed_sections: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    editorial_lines_removed: int = 0


def remove_author_block(lines: list, log: PreprocessingLog) -> list:
    """
    Elimina todo el contenido previo al inicio del cuerpo del paper,
    hasta la primera ocurrencia del resumen o introducción
    en las primeras 100 líneas.
    """
    # Analiza las primeras 100 líneas de texto para identifica el bloque de autor
    for i, line in enumerate(lines[:100]):
         # Si se identifica el bloque, se elimina y se añade al registro
        if INITIAL_PATTERN.match(line.strip()):
            log.removed_sections.append(
                f"Bloque de autor eliminado (líneas 0-{i}, {i} líneas eliminadas)"
            )
            return lines[i:]

    # Si no se identifica el bloque de autor en las primeras 100 líneas, se registra un aviso
    log.warnings.append(
        "No se encontró 'Abstract' ni 'Introduction' en las primeras 100 líneas — bloque de autor no eliminado"
    )
    return lines


def remove_section_from(lines: list, pattern_name: str, log: PreprocessingLog) -> list:
    """
    Elimina todo el contenido desde la primera línea que coincide con
    el patrón indicado hasta el final del documento.
    Registra un aviso si la sección no se encuentra.

    Estrategia por tipo de sección:
    - Apéndices y referencias bibliográficas: solo filtro por posición
      (debe estar en el 40% final del documento).
    - Resto de secciones: se exige línea anterior vacía.
    """
    pattern = PATTERNS[pattern_name]
    total_lines = len(lines)

    for i, line in enumerate(lines):
        # Si no se encuentra ningún patrón, se rompe el bucle
        if not pattern.match(line.strip()):
            continue

        position_pct = i / total_lines

        if pattern_name in ("appendix", "references", "bibliography"):
            # Para que los apendices y referencias sean eliminados, deben de
            # encontrarse en el 40% final del documento
            if position_pct < 0.6:
                continue
        else:
            # Para eliminar el resto de secciones, la linea de texto anterior
            # debe estar vacía
            prev_line = lines[i - 1].strip() if i > 0 else ""
            if prev_line != "":
                continue

        # Registrar secciones eliminadas y filtrar el texto
        log.removed_sections.append(
            f"{pattern_name} (desde línea {i}: '{line.strip()[:60]}')"
        )
        return lines[:i]

    # Si no se han detectado las secciones, se registra error
    log.warnings.append(f"Sección '{pattern_name}' no detectada")
    return lines


def remove_acknowledgments(lines: list, log: PreprocessingLog) -> list:
    """
    Elimina la sección de agradecimientos y declaraciones administrativas.
    A diferencia de referencias/apéndices, esta sección puede aparecer
    antes o después de las conclusiones, así que se elimina solo el bloque
    comprendido hasta la siguiente sección de nivel similar.
    """
    pattern = PATTERNS["acknowledgments"]
    # Patrón para detectar el inicio de la siguiente sección principal
    next_section = re.compile(r"^\s*(\d+\.?\s+)[A-Z][a-zA-Z\s]+$")

    result = []
    i = 0
    while i < len(lines):
        if pattern.match(lines[i].strip()):
            start = i
            i += 1
            # Avanzar hasta la siguiente sección o fin de documento
            while i < len(lines) and not next_section.match(lines[i].strip()):
                i += 1
            removed = i - start
            log.removed_sections.append(
                f"Agradecimientos eliminados (líneas {start}-{i}, {removed} líneas)"
            )
        else:
            result.append(lines[i])
            i += 1
    #Registrar aviso si no se detectó la sección de agradecimientos
    if not any("acknowledgments" in s for s in log.removed_sections):
        log.warnings.append("Sección de agradecimientos no detectada")

    return result


def remove_editorial_marks(lines: list, log: PreprocessingLog) -> list:
    """
    Elimina líneas que contienen marcas editoriales: marcas de agua,
    avisos de copyright, DOI,...
    """
    result = []
    removed = 0
    for line in lines:
        if any(p.search(line) for p in EDITORIAL_PATTERNS):
            removed += 1
        else:
            result.append(line)
    log.editorial_lines_removed = removed
    return result


def normalize_linebreaks(lines: list) -> list:
    """
    Une líneas que pertenecen al mismo párrafo pero están
    fragmentadas por la extracción de columnas del PDF.
    Heurística: una línea que no termina en punto, dos puntos,
    signo de interrogación o exclamación, y la siguiente empieza
    en minúscula, pertenecen al mismo párrafo.
    """
    result = []
    i = 0
    while i < len(lines):
        current = lines[i].rstrip()
        while (
            i + 1 < len(lines)
            and current
            and not re.search(r"[.!?:]\s*$", current)
            and lines[i + 1].strip()
            and lines[i + 1].strip()[0].islower()
        ):
            i += 1
            current = current + " " + lines[i].strip()
        result.append(current)
        i += 1
    return result

### 4.2. Pipeline principal de preprocesamiento

Función donde se ejecutan todas las funciones auxiliares definidas previamente para realizar el preprocesamiento de cada registro del corpus. Se invoca la clase `PreprocessingLog` para registrar todas las operaciones y avisos en cada etapa. Finalmente se devuelve el texto preprocesado como una variable de texto y el registro de operaciones.

Las etapas del pipeline son las siguientes:

1. Eliminar bloque de autores con `remove_author_block`
2. Eliminar disclaimers con `remove_section_from("disclaimers")`
3. Eliminar apéndices `remove_section_from("appendix")`
4. Eliminar referencias bibliográficas `remove_section_from("references")`
5. Eliminar agradecimientos y secciones administrativas con `remove_acknowledgments`
6. Eliminar marcas editoriales y cabeceras de revista con `remove_editorial_marks`
7. Normalizar saltos de línea con `normalize_linebreaks`
8. Eliminar múltiples saltos de línea que no fueron eliminados durante el pipeline de extracción o que hayan surgido al eliminar secciones.


In [ ]:
def preprocess_paper(text: str) -> tuple[str, PreprocessingLog]:
    """
    Aplica el pipeline completo de preprocesamiento sobre el texto
    de un artículo. Devuelve (texto_procesado, log).

    Orden de operaciones:
    1. Eliminar bloque inicial hasta Abstract o Introducción
    2. Eliminar disclaimers
    3. Eliminar apéndices
    4. Eliminar referencias bibliográficas
    5. Eliminar agradecimientos y secciones administrativas
    6. Eliminar marcas editoriales y cabeceras de revista
    7. Normalizar saltos de línea
    8. Limpieza final de espacios
    """
    log = PreprocessingLog()
    lines = text.split("\n")

    # Paso 1: bloque inicial
    lines = remove_author_block(lines, log)

    # Paso 2: disclaimer
    lines = remove_section_from(lines, "disclaimer", log)

    # Paso 3: apéndices
    lines = remove_section_from(lines, "appendix", log)

    # Paso 4: referencias
    lines = remove_section_from(lines, "references", log)
    if any("references" in w for w in log.warnings):
        lines = remove_section_from(lines, "bibliography", log)

    # Paso 5: agradecimientos y secciones administrativas
    lines = remove_acknowledgments(lines, log)

    # Paso 6: marcas editoriales y cabeceras de revista
    lines = remove_editorial_marks(lines, log)
    lines = [l for l in lines if not JOURNAL_HEADER_PATTERN.match(l.strip())]

    # Paso 7: normalizar saltos de línea
    lines = normalize_linebreaks(lines)

    # Paso 8: limpieza final
    text_out = "\n".join(lines)
    text_out = re.sub(r"\n{3,}", "\n\n", text_out)
    text_out = re.sub(r" {2,}", " ", text_out)
    text_out = text_out.strip()

    return text_out, log


print("✓ Pipeline definido")

## 5. Ejecución del pipeline de preprocesamiento sobre el corpus completo

Bucle principal que procesa los `N_PAPERS` registros del corpus de forma secuencial. Para cada registro, se construye la ruta al archivo `.txt` con el texto del documento correspondiente extraído y se preprocesa el texto mediante la función `preprocess_paper` definida anteriormente.

- Si el archivo no existe en la ruta especificada, se marca como `'ERROR'` con su motivo correspondiente
- Si el preprocesamiento se realiza exitosamente, el texto final se guarda como archivo `.json` en la ruta `PROCESSED_DIR` junto a los siguientes campos:
  - `paper_id`: Identificador del registro
  - `title`: Título del registro
  - `authors`: Autores
  - `year`: Año de publicación
  - `doi`: DOI del registro
  - `n_tokens`: Número de tokens del registro, estimado mediante `cl100k_base`
  - `text`: Texto preprocesado final como variable de texto

  Adicionalmente, se registran diferentes métricas durante el preprocesamiento para el posterior informe de preprocesamiento. Las métricas incluyen:
  - `paper_id`: ID del registro
  - `title`: Título del registro
  - `status`: Estado del preprocesamiento (`OK`. `WARNING`, `ERROR`)
  - `n_tokens_raw`: Número de tokens del texto extraído inicial
  - `n_tokens_processed`: Total de tokens tras el preprocesamiento
  - `tokens_removed`: Total de tokens eliminados durante el preprocesamiento
  - `pct_removed`: Porcentaje de tokens eliminados
  - `sections_removed`: Secciones eliminadas
  - `warnings`: Avisos registrados por `PreprocessingLog` durante el preprocesamiento
  - `error`: Errores registrados

  Finalmente, se imprime un resumen con el número de registros procesados correctamente, aquellos con avisos registrados durante el preprocesamiento, y los errores encontrados.

In [ ]:
report = []

for idx, row in df_meta.iterrows():
    # Construcción de la ruta al archivo con el texto extraído
    paper_id = row["paper_id"]
    title = row.get("title", paper_id) if "title" in df_meta.columns else paper_id
    short_title = str(title)[:55] + "..." if len(str(title)) > 55 else str(title)

    txt_path = os.path.join(RAW_DIR, f"{paper_id}.txt")

    # Métricas de preprocesamiento
    record = {
        "paper_id": paper_id,
        "title": title,
        "status": None,
        "n_tokens_raw": None,
        "n_tokens_processed": None,
        "tokens_removed": None,
        "pct_removed": None,
        "sections_removed": None,
        "warnings": None,
        "error": None,
    }

    # Recuperar tokens raw del informe de extracción
    extraction_row = df_extraction[df_extraction["paper_id"] == paper_id]
    if not extraction_row.empty:
        record["n_tokens_raw"] = int(extraction_row.iloc[0]["n_tokens"])

    # Error en caso de no encontrar el archivo
    if not os.path.exists(txt_path):
        record["status"] = "ERROR"
        record["error"] = ".txt no encontrado"
        print(f"  ✗ {paper_id} | .txt no encontrado")
        report.append(record)
        continue

    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        # Ejecución del pipeline de preprocesamiento
        processed_text, log = preprocess_paper(raw_text)

        n_tokens_processed = count_tokens(processed_text)
        n_tokens_raw = record["n_tokens_raw"] or count_tokens(raw_text)
        tokens_removed = n_tokens_raw - n_tokens_processed
        pct_removed = tokens_removed / n_tokens_raw * 100 if n_tokens_raw > 0 else 0

        # Consultar registro de avisos
        status = "WARNING" if log.warnings else "OK"

        # Guardar JSON
        output = {
            "paper_id": paper_id,
            "title": str(title),
            "authors": str(row.get("authors", "")) if "authors" in df_meta.columns else "",
            "year": str(row.get("year", "")) if "year" in df_meta.columns else "",
            "doi": str(row.get("doi", "")) if "doi" in df_meta.columns else "",
            "n_tokens": n_tokens_processed,
            "text": processed_text,
        }
        json_path = os.path.join(PROCESSED_DIR, f"{paper_id}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(output, f, ensure_ascii=False, indent=2)

        # Calcular métricas de preprocesamiento
        record.update({
            "status": status,
            "n_tokens_processed": n_tokens_processed,
            "tokens_removed": tokens_removed,
            "pct_removed": round(pct_removed, 1),
            "sections_removed": " | ".join(log.removed_sections),
            "warnings": " | ".join(log.warnings) if log.warnings else "",
        })

        # Impresión de cada registro y sus métricas de preprocesamiento,
        # indicando si el preprocesamiento ha sido exitoso o si se han registrado
        # avisos
        warn_tag = " ⚠" if log.warnings else ""
        print(f"  {'⚠' if warn_tag else '✓'} {paper_id} | "
              f"{n_tokens_raw:>7,} → {n_tokens_processed:>7,} tokens "
              f"(-{pct_removed:.1f}%){warn_tag} | {short_title}")

    # Impresión de errores durante el preprocesamiento
    except Exception as e:
        record["status"] = "ERROR"
        record["error"] = str(e)
        print(f"  ✗ {paper_id} | ERROR: {e}")

    report.append(record)

# Resumen final con el número de registros procesados correctamente, avisos y errores
print("\n" + "="*60)
print(f"OK:      {len([r for r in report if r['status'] == 'OK'])}")
print(f"WARNING: {len([r for r in report if r['status'] == 'WARNING'])}")
print(f"ERROR:   {len([r for r in report if r['status'] == 'ERROR'])}")

## 6. Output final

### 6.1. Informe de preprocesamiento

Se guarda un informe de preprocesamiento CSV en `REPORT_PATH` con las métricas de preprocesamiento de cada registro procesado.

In [ ]:
report_df = pd.DataFrame(report)
report_df.to_csv(REPORT_PATH, index=False, encoding="utf-8")
print(f"✓ Informe guardado en: {REPORT_PATH}")
report_df

### 6.2. Estadísticas comparativas pre/post preprocesamiento

Para los artículos procesados con éxito (`status = 'OK' o 'WARNING'`), se imprime una tabla comparativa con estadísticas del registro antes de ser preprocesado y tras serlo. Dichas estadísticas incluyen:

- Número de artículos procesados
- Total de tokens del corpus
- Media de tokens por artículo
- Mediana de tokens por artículo
- Número de tokens eliminados tras el pipeline de preprocesamiento
- Porcentaje de tokens eliminados

Adicionalmente, se muestran los 5 registros con mayor reducción en %, los registros en los que se han identificado avisos durante la ejecución del pipeline, y una lista de registros en los que se ha eliminado más de un 60% del contenido inicial, lo cual puede indicar un fallo del pipeline y que se requiera inspección manual del registro.

In [ ]:
ok = report_df[report_df["status"].isin(["OK", "WARNING"])].copy()

if len(ok) > 0:
    print("=" * 60)
    print("ESTADÍSTICAS COMPARATIVAS DEL CORPUS")
    print("=" * 60)
    print(f"  Artículos procesados:          {len(ok)}")
    print()
    print(f"  ANTES del preprocesamiento:")
    print(f"    Total tokens:                {ok['n_tokens_raw'].sum():>10,}")
    print(f"    Media tokens/artículo:       {ok['n_tokens_raw'].mean():>10,.0f}")
    print(f"    Mediana tokens/artículo:     {ok['n_tokens_raw'].median():>10,.0f}")
    print()
    print(f"  DESPUÉS del preprocesamiento:")
    print(f"    Total tokens:                {ok['n_tokens_processed'].sum():>10,}")
    print(f"    Media tokens/artículo:       {ok['n_tokens_processed'].mean():>10,.0f}")
    print(f"    Mediana tokens/artículo:     {ok['n_tokens_processed'].median():>10,.0f}")
    print()
    total_removed = ok['tokens_removed'].sum()
    pct_total = total_removed / ok['n_tokens_raw'].sum() * 100
    print(f"  REDUCCIÓN TOTAL:")
    print(f"    Tokens eliminados:           {total_removed:>10,}")
    print(f"    Reducción porcentual:        {pct_total:>9.1f}%")
    print()

    # Top 5 artículos con mayor reducción en %
    print("  TOP 5 ARTÍCULOS CON MAYOR REDUCCIÓN (%):")
    top5 = ok.nlargest(5, "tokens_removed")[["paper_id", "n_tokens_raw", "n_tokens_processed", "pct_removed"]]
    print(top5.to_string(index=False))
    print()

    # Artículos con avisos
    warnings = ok[ok["status"] == "WARNING"]
    if len(warnings) > 0:
        print(f"  ⚠ ARTÍCULOS CON AVISOS ({len(warnings)}):")
        for _, r in warnings.iterrows():
            print(f"    - {r['paper_id']}: {r['warnings']}")
    else:
        print("  ✓ Ningún artículo con avisos")

    # Artículos sospechosos: reducción > 60%
    sospechosos = ok[ok["pct_removed"] > 60]
    if len(sospechosos) > 0:
        print()
        print("  ⚠ ARTÍCULOS CON REDUCCIÓN > 60% (revisar manualmente):")
        for _, r in sospechosos.iterrows():
            print(f"    - {r['paper_id']}: -{r['pct_removed']}% | {r['sections_removed'][:80]}")

### 6.3. Inspección de registros

Se permite inspeccionar el texto preprocesado de cada registro y compararlo con el texto extraído del mismo registro en el pipeline de extracción, con el fin de realizar un análisis más específico acerca de la calidad del preprocesamiento. Cambia `PAPER_TO_INSPECT` con el ID del registro a inspeccionar, y `N_CHARS_PREVIEW` con el número de caracteres a mostrar del registro extraído.

In [ ]:
PAPER_TO_INSPECT = "paper_01"      # registro a inspeccionar
N_CHARS_PREVIEW = 2000             # número de caracteres a inspeccionar

json_path = os.path.join(PROCESSED_DIR, f"{PAPER_TO_INSPECT}.json")
txt_path  = os.path.join(RAW_DIR, f"{PAPER_TO_INSPECT}.txt")

if os.path.exists(json_path) and os.path.exists(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        raw = f.read()
    with open(json_path, "r", encoding="utf-8") as f:
        processed = json.load(f)

    print(f"{'='*60}")
    print(f"TEXTO RAW — primeros {N_CHARS_PREVIEW} caracteres")
    print(f"{'='*60}")
    print(raw[:N_CHARS_PREVIEW])

    print(f"\n{'='*60}")
    print(f"TEXTO PROCESADO — primeros {N_CHARS_PREVIEW} caracteres")
    print(f"{'='*60}")
    print(processed["text"][:N_CHARS_PREVIEW])

else:
    print(f"Archivos no encontrados para {PAPER_TO_INSPECT}")